# MNIST Deep ReLU Bias: Average Rank Over Epochs

This notebook pulls MNIST deep ReLU sweep results directly from Weights & Biases (W&B) and
computes the average low-rank bias **across training epochs** for every run in a sweep.
Specify the sweep you care about, authenticate with W&B (e.g. `wandb login`), and execute the cells
to build a plot of epoch-averaged ranks as a function of batch size for each initial learning rate.

In [1]:
from __future__ import annotations

import logging
from pathlib import Path
from typing import Iterable
# add parent directory to sys.path for local imports
import os
import sys
sys.path.append(os.path.expanduser("~/inductive-bias/"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb
from wandb.apis.public import Api, Run
from lightning import LightningModule

# --- Configure which sweep to load -------------------------------------------------------------
WANDB_ENTITY = "jhrudoler-penn"
WANDB_PROJECT = "inductive-bias"
SWEEP_ID = "1s1i8plu"  # e.g. "lkcv7u6p"

SWEEP_PATH = f"{WANDB_ENTITY}/{WANDB_PROJECT}/{SWEEP_ID}"
if any(token.startswith('replace-with') for token in (WANDB_ENTITY, SWEEP_ID)):
    raise ValueError('Update WANDB_ENTITY and SWEEP_ID before continuing.')

# --- Matplotlib style and output directories ----------------------------------------------------
REPO_ROOT_CANDIDATES: list[Path] = [Path.cwd(), Path.cwd().parent]
REPO_ROOT: Path | None = None
for candidate in REPO_ROOT_CANDIDATES:
    if (candidate / 'clean_fig.mplstyle').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise FileNotFoundError('Could not locate clean_fig.mplstyle relative to the working directory.')

STYLE_PATH = REPO_ROOT / 'clean_fig.mplstyle'
FIGURES_DIR = REPO_ROOT / 'results' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use(STYLE_PATH)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
LOGGER = logging.getLogger('mnist_average_rank')
LOGGER.info('Using sweep %s', SWEEP_PATH)


2025-10-29 05:02:54,535 | INFO | mnist_average_rank | Using sweep jhrudoler-penn/inductive-bias/1s1i8plu


In [ ]:
# Authenticate (uses environment WANDB_API_KEY if already set) and fetch sweep metadata.
api = Api()
sweep = api.sweep(SWEEP_PATH)
runs = list(sweep.runs)
LOGGER.info('Fetched %d runs from sweep %s', len(runs), SWEEP_PATH)
[r.id for r in runs]


2025-10-29 04:55:53,288 | INFO | mnist_average_rank | Fetched 18 runs from sweep jhrudoler-penn/inductive-bias/1s1i8plu


['lktoxmti',
 'twzx6qs1',
 'd99wz9s9',
 '9m13ibek',
 '5ht3r1a9',
 'xp2e4syb',
 '9nvoonhf',
 'k4gseg7n',
 '1hcwpr2o',
 'jazrdd13',
 '0p7bx37i',
 'ucl7egku',
 '03m261mb',
 '7zxpoyx0',
 'aiot7juf',
 'y1cwxfnc',
 'ywjhfhiq',
 'ug6mtvhq']

In [ ]:
def _unwrap_config_value(value: object) -> object:
    if isinstance(value, dict) and 'value' in value and len(value) == 1:
        return value['value']
    return value


def _to_int(value: object) -> int | None:
    try:
        return int(value) if value is not None else None
    except (TypeError, ValueError):
        return None


def _to_float(value: object) -> float | None:
    try:
        return float(value) if value is not None else None
    except (TypeError, ValueError):
        return None


def extract_run_metadata(run: Run) -> dict[str, object]:
    config = run.config

    def fetch(*keys: str) -> object | None:
        for key in keys:
            if key in config:
                return _unwrap_config_value(config[key])
        return None

    batch_size = _to_int(fetch('batch_size', 'batch-size'))
    learning_rate = _to_float(fetch('lr', 'learning_rate'))
    depth = _to_int(fetch('depth'))

    return {
        'run_id': run.id,
        'run_name': run.name,
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'depth': depth,
    }


def collect_run_epoch_ranks(run: Run, *, layer_count_hint: int | None = None) -> pd.DataFrame:
    history = run.history(samples=None, pandas=True)
    if history is None or history.empty:
        LOGGER.warning('Run %s has no recorded history; skipping.', run.id)
        return pd.DataFrame()

    epoch_column = 'epoch' if 'epoch' in history.columns else 'trainer/global_step'
    if epoch_column not in history.columns:
        LOGGER.warning('Run %s lacks an epoch indicator; skipping.', run.id)
        return pd.DataFrame()

    layer_columns = [col for col in history.columns if col.startswith('bias/low_rank_per_layer/')]
    total_key = 'bias/low_rank_total'

    if not layer_columns:
        if total_key not in history.columns:
            LOGGER.warning('Run %s does not expose low-rank metrics; skipping.', run.id)
            return pd.DataFrame()
        if layer_count_hint is None:
            LOGGER.warning('Run %s requires a layer_count_hint to average ranks; skipping.', run.id)
            return pd.DataFrame()
        total_series = pd.to_numeric(history[total_key], errors='coerce')
        history['average_rank'] = total_series / layer_count_hint
    else:
        layer_values = history[layer_columns].apply(pd.to_numeric, errors='coerce')
        history['average_rank'] = layer_values.mean(axis=1)

    history = history.dropna(subset=[epoch_column, 'average_rank'])
    if history.empty:
        LOGGER.warning('Run %s has no valid average rank data; skipping.', run.id)
        return pd.DataFrame()

    history['epoch'] = pd.to_numeric(history[epoch_column], errors='coerce')
    history = history.dropna(subset=['epoch'])
    if history.empty:
        LOGGER.warning('Run %s has no valid epochs; skipping.', run.id)
        return pd.DataFrame()

    history['epoch'] = history['epoch'].astype(int)

    epoch_frame = (
        history.groupby('epoch', as_index=False)
        .agg(average_rank=('average_rank', 'mean'))
        .sort_values('epoch')
        .reset_index(drop=True)
    )
    epoch_frame['run_id'] = run.id
    epoch_frame['run_name'] = run.name
    return epoch_frame


In [ ]:
# load model artifacts from the runs in the sweep
all_models = {}
for run in runs:
    model_artifacts = []
    print(len(run.logged_artifacts()))
    for artifact in run.logged_artifacts():
        if artifact.type == 'model' and ('latest' in artifact.aliases):
            model_artifacts.append(artifact)
    all_models[run.id] = model_artifacts
    LOGGER.info('Run %s has %d model artifacts.', run.id, len(model_artifacts))

2025-10-29 04:56:07,279 | INFO | mnist_average_rank | Run lktoxmti has 1 model artifacts.


2


2025-10-29 04:56:07,749 | INFO | mnist_average_rank | Run twzx6qs1 has 1 model artifacts.


3
3


2025-10-29 04:56:08,274 | INFO | mnist_average_rank | Run d99wz9s9 has 1 model artifacts.


...

2025-10-29 04:56:14,743 | INFO | mnist_average_rank | Run y1cwxfnc has 1 model artifacts.


2


2025-10-29 04:56:15,185 | INFO | mnist_average_rank | Run ywjhfhiq has 1 model artifacts.


1


2025-10-29 04:56:15,602 | INFO | mnist_average_rank | Run ug6mtvhq has 0 model artifacts.


In [ ]:
from lightning import LightningModule
# load the first run's best model artifact
first_run = runs[0]
best_model_artifact = None
for artifact in first_run.logged_artifacts():
    if artifact.type == 'model' and ('best' in artifact.aliases):
        best_model_artifact = artifact
        break
# get the LightningModule from the artifact
datadir = best_model_artifact.download()
lightning_module = LightningModule.load_from_checkpoint(datadir / 'model.ckpt')

wandb:   1 of 1 files downloaded.  


NameError: name 'LightningModule' is not defined

NameError: name 'best_model_artifact' is not defined

In [56]:
sys.path.append('/home/jrudoler/inductive-bias')
from experiments.mnist_deep_relu_bias import DeepReLULightningModule
module = DeepReLULightningModule.load_from_checkpoint(datadir / 'model.ckpt')

TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [9]:
epoch_frames: list[pd.DataFrame] = []
run_summaries: list[dict[str, object]] = []

for run in runs:
    metadata = extract_run_metadata(run)
    layer_hint = (metadata['depth'] + 1) if isinstance(metadata['depth'], int) else None
    epoch_df = collect_run_epoch_ranks(run, layer_count_hint=layer_hint)
    if epoch_df.empty:
        LOGGER.warning('Skipping run %s (%s) due to missing epoch data.', run.id, run.name)
        continue
    if metadata['batch_size'] is None or metadata['learning_rate'] is None:
        LOGGER.warning('Skipping run %s (%s) due to missing hyperparameters.', run.id, run.name)
        continue

    epoch_df = epoch_df.assign(
        batch_size=metadata['batch_size'],
        learning_rate=metadata['learning_rate'],
    )
    epoch_frames.append(epoch_df)

    mean_rank = float(epoch_df['average_rank'].mean())
    std_rank = float(epoch_df['average_rank'].std(ddof=0)) if len(epoch_df) > 1 else 0.0

    run_summaries.append(
        {
            'run_id': metadata['run_id'],
            'run_name': metadata['run_name'],
            'batch_size': metadata['batch_size'],
            'learning_rate': metadata['learning_rate'],
            'epochs_logged': int(epoch_df['epoch'].nunique()),
            'average_rank_over_epochs': mean_rank,
            'average_rank_over_epochs_std': std_rank,
        }
    )

if not run_summaries:
    raise ValueError('No runs with usable epoch-level rank data were collected.')

epoch_level_df = pd.concat(epoch_frames, ignore_index=True)
run_summary_df = pd.DataFrame(run_summaries).sort_values(['learning_rate', 'batch_size', 'run_id']).reset_index(drop=True)
run_summary_df


2025-10-28 09:59:21,769 | WARNING | mnist_average_rank | Skipping run lktoxmti (cosmic-sweep-18) due to missing epoch data.
2025-10-28 09:59:22,087 | WARNING | mnist_average_rank | Skipping run twzx6qs1 (woven-sweep-17) due to missing epoch data.
2025-10-28 09:59:22,432 | WARNING | mnist_average_rank | Skipping run d99wz9s9 (misty-sweep-16) due to missing epoch data.
2025-10-28 09:59:22,722 | WARNING | mnist_average_rank | Skipping run 9m13ibek (helpful-sweep-15) due to missing epoch data.
2025-10-28 09:59:23,045 | WARNING | mnist_average_rank | Skipping run 5ht3r1a9 (rare-sweep-14) due to missing epoch data.
2025-10-28 09:59:23,286 | WARNING | mnist_average_rank | Skipping run xp2e4syb (magic-sweep-13) due to missing epoch data.
2025-10-28 09:59:23,584 | WARNING | mnist_average_rank | Skipping run 9nvoonhf (lyric-sweep-12) due to missing epoch data.
2025-10-28 09:59:23,824 | WARNING | mnist_average_rank | Skipping run k4gseg7n (efficient-sweep-11) due to missing epoch data.
2025-10-28

ValueError: No runs with usable epoch-level rank data were collected.

In [ ]:
aggregate_df = (
    run_summary_df
    .groupby(['learning_rate', 'batch_size'], as_index=False)
    .agg(
        runs=('run_id', 'count'),
        average_rank=('average_rank_over_epochs', 'mean'),
        rank_std=('average_rank_over_epochs', 'std'),
    )
    .sort_values(['learning_rate', 'batch_size'])
)
aggregate_df['rank_std'] = aggregate_df['rank_std'].fillna(0.0)
aggregate_df


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for lr_value, subset in aggregate_df.groupby('learning_rate'):
    subset_sorted = subset.sort_values('batch_size')
    ax.errorbar(
        subset_sorted['batch_size'],
        subset_sorted['average_rank'],
        yerr=subset_sorted['rank_std'],
        marker='o',
        capsize=4,
        label=f'lr = {lr_value:g}',
    )

ax.set_xlabel('Batch size')
ax.set_ylabel('Epoch-averaged rank across layers')
ax.set_title('MNIST Deep ReLU Bias: Average Rank vs Batch Size')
ax.set_xscale('log', base=2)
ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
ax.legend(title='Initial learning rate')

figure_path = FIGURES_DIR / f'{SWEEP_ID}_average_rank_vs_batch_size.pdf'
fig.tight_layout()
fig.savefig(figure_path, dpi=300)
figure_path


The plot is saved to `results/figures/{SWEEP_ID}_average_rank_vs_batch_size.pdf`.